In [1]:
import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401
from dask.distributed import Client as DaskClient
from odc.stac import configure_s3_access
from sklearn.ensemble import RandomForestClassifier

from utils import BBOX, BBOX_GDF, BBOX_GEOMETRY, DATETIME, load_data

In [2]:
%reload_ext autoreload
%autoreload 2

## Find and load S2 data

Load data and set up your array to use for prediction

In [3]:
# Create local dask cluster to improve data load time. Only run this once.
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit="16GB")

# Configure S3 access. Cloud defaults is an optimisation,
# TODO: Is requester pays required?
configure_s3_access(cloud_defaults=True, requester_pays=True)

In [4]:
BBOX_GDF.explore()

In [5]:
merged = load_data(BBOX, DATETIME)

Found Sentinel-2 items: 4
Found DEM items: 1


2026-10-23 16:23:32,272 - distributed.worker - ERROR - Compute Failed
Key:       ('open-data-972e3d99489a88f96a693dcf0ff2032a', 0)
State:     executing
Task:  <Task ('open-data-972e3d99489a88f96a693dcf0ff2032a', 0) _dask_open_reader(...)>
Exception: 'AttributeError("\'NoneType\' object has no attribute \'Session\'")'
Traceback: '  File "/Users/wj/Projects/dep/minerals-model/.venv/lib/python3.15/site-packages/odc/loader/_builder.py", line 460, in _dask_open_reader\n    with rdr.restore_env(env, load_state) as ctx:\n         ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^\n  File "/opt/homebrew/Cellar/python@3.15/3.15.0/Frameworks/Python.framework/Versions/3.15/lib/python3.15/contextlib.py", line 203, in __enter__\n    return next(self.gen)\n  File "/Users/wj/Projects/dep/minerals-model/.venv/lib/python3.15/site-packages/odc/loader/_rio.py", line 184, in restore_env\n    with rio_env(**env):\n         ~~~~~~~^^^^^^^\n  File "/Users/wj/Projects/dep/minerals-model/.venv/lib/python3.15/site-packages/odc/l

AttributeError: 'DataArray' object has no attribute 'values'

In [ ]:
merged

## Train and predict

When you change your training data, you can re-train and predict here.

In [ ]:
training_file = "training_data/mred-tdata.geojson"  # TODO: Check which training data to use training_data/MRD_dissagregated_32.geojson?

tdata = gpd.read_file(training_file, bbox=BBOX_GEOMETRY)
tdata = tdata.set_geometry("geometry")
tdata = tdata.to_crs(merged.rio.crs)

print(tdata.crs)
print(merged.rio.crs)
if tdata.crs != merged.rio.crs:
    tdata = tdata.to_crs(merged.rio.crs)

tdata.explore()

In [ ]:
len(tdata)

In [ ]:
tdata["lulc_code"].value_counts().sort_index()

In [ ]:
# 1. Ensure both tables are aligned correctly using the shared 'index' column
# This prevents positional or index-label mismatches from creating NaN rows
pt_values = None  # TODO: Figure out what pt_values should be
training_array = pd.merge(tdata, pt_values, on="index", suffixes=("_tdata", "_pt"))

# 2. If tdata is a GeoDataFrame, ensure geometry is preserved from tdata
# TODO: Just use 1 data type for training_array. Remove conditional logic
if not isinstance(training_array, gpd.GeoDataFrame):
    training_array = gpd.GeoDataFrame(
        training_array, geometry=tdata.geometry, crs=tdata.crs
    )

# 3. Reproject the spatial GeoDataFrame to WGS84 (EPSG:4326) safely
training_array = training_array.to_crs(4326)

# 4. Drop unwanted metadata, coordinate, and duplicate columns cleanly
cols_to_drop = [
    "y_tdata",
    "x_tdata",
    "y_pt",
    "x_pt",
    "y",
    "x",
    "spatial_ref",
    "time",
    "fid",
    "index",
    "lulc_class",
    "path",
    "geometry",
    "layer",
    "id",
    "Unnamed: 0",
]

# Only drop columns that actually exist in the dataframe to avoid KeyErrors
existing_cols_to_drop = [col for col in cols_to_drop if col in training_array.columns]
training_array = training_array.drop(columns=existing_cols_to_drop)

# 5. Check final shape and clean NaNs safely
print("Final training array shape:", training_array.shape)
training_array = training_array.dropna(subset=["lulc_code"])

In [ ]:
len(training_array)

In [ ]:
training_array

In [ ]:
classifier = RandomForestClassifier(
    # TODO: Ensure features are named
    # TODO: Tune hyperparameters for better performance
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42,
)

training_data = np.array(training_array)[:, 1:]
classes = np.array(training_array)[:, 0]

model = classifier.fit(training_data, classes)

In [ ]:
joblib.dump(model, "test_model.dump")

In [ ]:
# Print feature importances against column headings
fields_importances = sorted(
    zip(training_array.columns[1:], classifier.feature_importances_),
    key=lambda x: x[1],
    reverse=True,
)

for i in fields_importances:
    print(f"{i[0]:<11}| {i[1]:.2f}")